In [ ]:
from google.colab import files
import os

files.upload()  # select kaggle.json

os.makedirs('/root/.config/kaggle', exist_ok=True)
os.rename('kaggle.json', '/root/.config/kaggle/kaggle.json')
os.chmod('/root/.config/kaggle/kaggle.json', 0o600)

!pip install -q kaggle
print("Kaggle API ready")

In [ ]:
!kaggle datasets download -d vbookshelf/v2-plant-seedlings-dataset -p /content/data --unzip -q
print("Download complete")

In [ ]:
import os

RAW_DIR = '/content/data'
EXCLUDE = {'nonsegmentedv2'}

classes = sorted([
    d for d in os.listdir(RAW_DIR)
    if os.path.isdir(os.path.join(RAW_DIR, d)) and d not in EXCLUDE
])

print(f"Classes ({len(classes)}):")
total = 0
for cls in classes:
    n = len(os.listdir(os.path.join(RAW_DIR, cls)))
    total += n
    print(f"  {cls:<35} {n:>4} images")
print(f"\nTotal images: {total}")

In [ ]:
import os
import shutil
import random

SPLIT_DIR = '/content/split'
TRAIN_DIR = os.path.join(SPLIT_DIR, 'train')
VAL_DIR   = os.path.join(SPLIT_DIR, 'val')
TEST_DIR  = os.path.join(SPLIT_DIR, 'test')

VAL_FRAC  = 0.15
TEST_FRAC = 0.15
SEED      = 42
random.seed(SEED)

for split in [TRAIN_DIR, VAL_DIR, TEST_DIR]:
    for cls in classes:
        os.makedirs(os.path.join(split, cls), exist_ok=True)

split_counts = {}
for cls in classes:
    imgs = sorted(os.listdir(os.path.join(RAW_DIR, cls)))
    random.shuffle(imgs)
    n = len(imgs)
    n_val  = int(n * VAL_FRAC)
    n_test = int(n * TEST_FRAC)
    n_train = n - n_val - n_test

    splits = [
        (imgs[:n_train],              TRAIN_DIR),
        (imgs[n_train:n_train+n_val], VAL_DIR),
        (imgs[n_train+n_val:],        TEST_DIR),
    ]
    for file_list, dest_root in splits:
        for f in file_list:
            shutil.copy(
                os.path.join(RAW_DIR, cls, f),
                os.path.join(dest_root, cls, f)
            )
    split_counts[cls] = (n_train, n_val, n_test)

print(f"{'Class':<35} {'Train':>6} {'Val':>6} {'Test':>6}")
print('-' * 58)
for cls, (tr, v, te) in split_counts.items():
    print(f"{cls:<35} {tr:>6} {v:>6} {te:>6}")
total_tr = sum(v[0] for v in split_counts.values())
total_v  = sum(v[1] for v in split_counts.values())
total_te = sum(v[2] for v in split_counts.values())
print('-' * 58)
print(f"{'TOTAL':<35} {total_tr:>6} {total_v:>6} {total_te:>6}")

In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.utils.class_weight import compute_class_weight

IMG_SIZE   = (224, 224)
BATCH_SIZE = 32

# Training: augmented, no rescale
train_datagen = ImageDataGenerator(
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.15,
    zoom_range=0.2,
    horizontal_flip=True,
    vertical_flip=True,
    brightness_range=[0.8, 1.2],
    fill_mode='nearest'
)

# Val / Test: raw pixels only
plain_datagen = ImageDataGenerator()

train_generator = train_datagen.flow_from_directory(
    TRAIN_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=classes,
    shuffle=True,
    seed=SEED
)

val_generator = plain_datagen.flow_from_directory(
    VAL_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=classes,
    shuffle=False
)

test_generator = plain_datagen.flow_from_directory(
    TEST_DIR,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='categorical',
    classes=classes,
    shuffle=False
)

print(f"Train: {train_generator.samples} | Val: {val_generator.samples} | Test: {test_generator.samples}")

# Class weights from training labels only
train_labels = train_generator.classes
cw_array = compute_class_weight('balanced', classes=np.unique(train_labels), y=train_labels)
class_weights = dict(enumerate(cw_array))

print("\nClass weights (higher = rarer class):")
for idx, cls in enumerate(classes):
    print(f"  {cls:<35} {class_weights[idx]:.4f}")

In [ ]:
import matplotlib.pyplot as plt

train_generator.reset()
images, labels = next(train_generator)

# Normalise to [0, 1] for display only
display_imgs = np.clip(images / 255.0, 0, 1)

fig, axes = plt.subplots(4, 8, figsize=(18, 9))
for i, ax in enumerate(axes.flatten()):
    ax.imshow(display_imgs[i])
    ax.set_title(classes[np.argmax(labels[i])], fontsize=7)
    ax.axis('off')

plt.suptitle('Augmented training batch (raw [0–255] — display only normalised)', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras import layers, Model
import tensorflow as tf

NUM_CLASSES = len(classes)

def build_model(num_classes):
    base_model = EfficientNetB0(
        weights='imagenet',
        include_top=False,
        input_shape=(224, 224, 3)
    )
    base_model.trainable = False

    inputs = tf.keras.Input(shape=(224, 224, 3))
    x = tf.keras.applications.efficientnet.preprocess_input(inputs)  # [0,255] -> normalised
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.4)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(num_classes, activation='softmax')(x)

    return Model(inputs, outputs), base_model

model, base_model = build_model(NUM_CLASSES)
trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
total     = sum(tf.size(w).numpy() for w in model.weights)
print(f"Trainable params (Phase 1): {trainable:,} / {total:,} total")
model.summary(line_length=90)

In [ ]:
from tensorflow.keras import optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint

callbacks_p1 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-6, verbose=1),
    ModelCheckpoint('best_phase1.keras', monitor='val_loss', save_best_only=True, verbose=1)
]

model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("Phase 1: training head only (base frozen)...\n")
history1 = model.fit(
    train_generator,
    epochs=20,
    validation_data=val_generator,
    class_weight=class_weights,
    callbacks=callbacks_p1
)

In [ ]:
# Unfreeze top 50 layers of the backbone
base_model.trainable = True
for layer in base_model.layers[:-50]:
    layer.trainable = False

trainable = sum(tf.size(w).numpy() for w in model.trainable_weights)
print(f"Trainable params (Phase 2): {trainable:,}")

callbacks_p2 = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.3, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint('best_phase2.keras', monitor='val_loss', save_best_only=True, verbose=1)
]

# Recompile required after changing trainable status
model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

print("\nPhase 2: fine-tuning top 50 layers...\n")
history2 = model.fit(
    train_generator,
    epochs=25,
    validation_data=val_generator,
    class_weight=class_weights,
    callbacks=callbacks_p2
)

In [ ]:
import matplotlib.pyplot as plt

def plot_history(h1, h2):
    acc     = h1.history['accuracy']     + h2.history['accuracy']
    val_acc = h1.history['val_accuracy'] + h2.history['val_accuracy']
    loss    = h1.history['loss']         + h2.history['loss']
    val_loss = h1.history['val_loss']    + h2.history['val_loss']
    phase1_end = len(h1.history['accuracy'])
    epochs = range(1, len(acc) + 1)

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(epochs, acc,     label='Train Accuracy')
    ax1.plot(epochs, val_acc, label='Val Accuracy')
    ax1.axvline(x=phase1_end, color='gray', linestyle='--', label='Fine-tune start')
    ax1.set_title('Accuracy'); ax1.set_xlabel('Epoch'); ax1.legend()

    ax2.plot(epochs, loss,     label='Train Loss')
    ax2.plot(epochs, val_loss, label='Val Loss')
    ax2.axvline(x=phase1_end, color='gray', linestyle='--', label='Fine-tune start')
    ax2.set_title('Loss'); ax2.set_xlabel('Epoch'); ax2.legend()

    plt.suptitle('Training History — Phase 1 + Phase 2', fontsize=13)
    plt.tight_layout()
    plt.show()

plot_history(history1, history2)

In [ ]:
import numpy as np
from sklearn.metrics import classification_report, accuracy_score

test_generator.reset()
y_pred_probs = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes[:len(y_pred)]

print(f"\nTest samples evaluated: {len(y_true)}")
print(f"Overall Test Accuracy:  {accuracy_score(y_true, y_pred) * 100:.2f}%\n")
print("Per-class breakdown:")
print(classification_report(y_true, y_pred, target_names=classes, digits=3))

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_true, y_pred)
cm_pct = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100

fig, ax = plt.subplots(figsize=(13, 10))
sns.heatmap(
    cm_pct, annot=True, fmt='.1f',
    xticklabels=classes, yticklabels=classes,
    cmap='Blues', linewidths=0.5, ax=ax
)
ax.set_xlabel('Predicted', fontsize=12)
ax.set_ylabel('True', fontsize=12)
ax.set_title('Confusion Matrix — Test Set (% per true class)', fontsize=13)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import tensorflow as tf
import matplotlib.pyplot as plt
import cv2

# Build a sub-model that outputs (last conv activations, final predictions)
# EfficientNetB0's last conv layer before pooling is 'top_conv'
base        = model.get_layer('efficientnetb0')
top_conv    = base.get_layer('top_conv')

# Sub-model: input image -> top_conv activations
conv_model = tf.keras.Model(
    inputs=model.inputs,
    outputs=[top_conv.output, model.output]  # won't work directly due to nesting
)

# Correct approach: build grad model using the full model's computation graph
# by creating an intermediate model up to top_conv within base, then chaining

base_input  = base.input
conv_output = top_conv.output
base_conv_model = tf.keras.Model(inputs=base_input, outputs=conv_output)

def compute_gradcam(img_array, class_idx=None):
    """
    Compute Grad-CAM heatmap for a single image.
    img_array: (224, 224, 3) uint8 or float32 in [0, 255]
    class_idx: if None, uses the predicted class
    Returns: heatmap (H, W), predictions array, predicted class index
    """
    img_tensor = tf.cast(img_array[np.newaxis, ...], tf.float32)  # (1, 224, 224, 3)
    preprocess = tf.keras.applications.efficientnet.preprocess_input

    with tf.GradientTape() as tape:
        # Preprocess then get conv activations from base
        x_pre = preprocess(img_tensor)
        conv_out = base_conv_model(x_pre, training=False)   # (1, H, W, C)
        tape.watch(conv_out)

        # Run the rest of the base after top_conv, then the head
        x = conv_out
        found = False
        for layer in base.layers:
            if found:
                x = layer(x, training=False)
            if layer.name == 'top_conv':
                found = True

        # Run the custom head (everything after efficientnetb0 in outer model)
        head_started = False
        for layer in model.layers:
            if head_started:
                x = layer(x, training=False)
            if layer.name == 'efficientnetb0':
                head_started = True

        preds = x  # (1, num_classes)
        pred_idx = class_idx if class_idx is not None else int(tf.argmax(preds[0]))
        score = preds[:, pred_idx]

    grads = tape.gradient(score, conv_out)            # (1, H, W, C)
    pooled = tf.reduce_mean(grads, axis=(0, 1, 2))   # (C,)
    heatmap = conv_out[0] @ pooled[..., tf.newaxis]  # (H, W, 1)
    heatmap = tf.squeeze(heatmap)                     # (H, W)
    heatmap = tf.nn.relu(heatmap)
    heatmap = heatmap / (tf.reduce_max(heatmap) + 1e-8)
    return heatmap.numpy(), preds.numpy()[0], pred_idx


def overlay_gradcam(img_rgb, heatmap, alpha=0.45):
    """Resize heatmap and overlay as a JET colormap on the original image."""
    h, w = img_rgb.shape[:2]
    hm_resized = cv2.resize(heatmap, (w, h))
    hm_uint8   = np.uint8(255 * hm_resized)
    hm_color   = cv2.applyColorMap(hm_uint8, cv2.COLORMAP_JET)
    hm_color   = cv2.cvtColor(hm_color, cv2.COLOR_BGR2RGB)
    overlay    = np.clip(hm_color * alpha + img_rgb * (1 - alpha), 0, 255).astype(np.uint8)
    return overlay

print("Grad-CAM functions defined.")

In [ ]:
import random
from tensorflow.keras.utils import load_img, img_to_array

# Sample one image per class from the test set
sample_paths = []
for cls in classes:
    cls_dir = os.path.join(TEST_DIR, cls)
    img_file = random.choice(os.listdir(cls_dir))
    sample_paths.append((cls, os.path.join(cls_dir, img_file)))

n = len(sample_paths)
fig, axes = plt.subplots(n, 2, figsize=(8, n * 3.2))

for i, (true_cls, img_path) in enumerate(sample_paths):
    img = load_img(img_path, target_size=(224, 224))
    img_array = img_to_array(img)  # float32 [0, 255]
    img_rgb   = img_array.astype(np.uint8)

    heatmap, preds, pred_idx = compute_gradcam(img_array)
    overlaid = overlay_gradcam(img_rgb, heatmap)

    pred_cls   = classes[pred_idx]
    confidence = preds[pred_idx] * 100
    correct    = '✓' if pred_cls == true_cls else '✗'

    axes[i, 0].imshow(img_rgb)
    axes[i, 0].set_title(f"True: {true_cls}", fontsize=8)
    axes[i, 0].axis('off')

    axes[i, 1].imshow(overlaid)
    axes[i, 1].set_title(f"{correct} Pred: {pred_cls} ({confidence:.1f}%)", fontsize=8)
    axes[i, 1].axis('off')

plt.suptitle('Grad-CAM — Original vs Saliency Map (one per class)', fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
from tensorflow.keras.utils import load_img, img_to_array

SAVE_PATH = '/content/crop_weed_classifier_final.keras'
model.save(SAVE_PATH)
print(f"Model saved to {SAVE_PATH}")

model.export('/content/crop_weed_savedmodel')
print("SavedModel exported to /content/crop_weed_savedmodel")

# Sanity check: reload and run one prediction
reloaded = tf.keras.models.load_model(SAVE_PATH)

sample_cls  = classes[0]
sample_dir  = os.path.join(TEST_DIR, sample_cls)
sample_file = os.listdir(sample_dir)[0]
img = load_img(os.path.join(sample_dir, sample_file), target_size=(224, 224))
arr = img_to_array(img)[np.newaxis, ...]  # (1, 224, 224, 3)

pred = reloaded.predict(arr, verbose=0)
pred_cls = classes[np.argmax(pred[0])]
print(f"\nReload sanity check — True: {sample_cls} | Predicted: {pred_cls} ({np.max(pred)*100:.1f}%)")